In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
def prepare_mlp_data(file_path):
    df = pd.read_csv(file_path)

    # Convert critical columns to numeric, coercing errors to NaN
    df['Open'] = pd.to_numeric(df['Open'], errors='coerce')
    df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
    df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')

    # Regex to extract Strike from Ticker (e.g., AARTIIND28NOV24490PE.NFO)
    # The study emphasizes Strike as a critical input for MLP learning
    def parse_strike(ticker):
        # Corrected regex: r"24(\d+)[CP]E" should be r"24(\d+)[CP]E"
        match = re.search(r"24(\d+)[CP]E", str(ticker))
        return float(match.group(1)) if match else None

    df['Strike'] = df['Ticker'].apply(parse_strike)
    df = df.dropna(subset=['Strike', 'Close', 'Open', 'Volume']) # Drop rows where Strike, Close, Open, or Volume are NaN

    # Filter out rows where Strike is 0 to avoid division by zero in S_K
    df = df[df['Strike'] != 0]

    # Feature Engineering based on paper inputs: S/K, T, and Volume
    df['S_K'] = df['Open'] / df['Strike']

    # Converting Expiry/Date for Time to Maturity (T)
    # The MLP requires T to capture the time-value decay
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    # Extracting expiry date, coercing errors to NaT (Not a Time)
    # Corrected regex: r'(\\d{2}[A-Z]{3}\\d{2})' should be r'(\d{2}[A-Z]{3}\d{2})'
    df['Expiry_dt'] = pd.to_datetime(df['Ticker'].str.extract(r'(\d{2}[A-Z]{3}\d{2})')[0], format='%d%b%y', errors='coerce')

    # Drop rows where Expiry_dt is NaT before calculating T
    df = df.dropna(subset=['Expiry_dt'])

    df['T'] = (df['Expiry_dt'] - df['Date_dt']).dt.days / 365.0
    df = df[df['T'] >= 0] # Filter out expired contracts, also handles NaNs from previous steps

    # Features: Moneyness (S/K), Time (T), and Volume
    features = ['S_K', 'T', 'Volume']
    X = df[features].values
    y = (df['Close'] / df['Strike']).values # Normalized Target to stabilize ANN weights

    # Ensure X and y do not contain NaN or infinite values before scaling
    # Drop rows from X and y if any of the feature columns are non-finite
    finite_rows_mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X[finite_rows_mask]
    y = y[finite_rows_mask]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
def build_mlp(input_dim):
    # Standard feedforward structure with non-linear activation
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='linear')
    ])

    # Using MSE as the loss function to optimize for regression metrics
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
X_train, X_test, y_train, y_test = prepare_mlp_data('NSE_FNO_DATA_2024-10-21.CSV')
model = build_mlp(X_train.shape[1])

# Training with Early Stopping to prevent overfitting to market noise
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1, callbacks=[early_stop], verbose=0)

# Predictions
y_pred = model.predict(X_test).flatten()

# Calculations for MAE, RMSE, and R2
mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred)**2)
rmse = np.sqrt(mse)
r2 = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2))

print(f"--- MLP PERFORMANCE (NSE DATASET) ---")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.4f}")

7432/7432 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step
--- MLP PERFORMANCE (NSE DATASET) ---
MAE:  0.000241
RMSE: 0.000627
R2:   0.9986
